Question 5: Async Data Pipeline

In [1]:
import pandas as pd
import sqlite3
import time
import aiosqlite
import asyncio

Using the CSV file from Question 1, filtering the data to include only 'Copper' and 'Zinc' for the year 2020 & 2021:

In [2]:
df = pd.read_csv("../data/MarketData.csv", skiprows=6)

df["Dates"] = pd.to_datetime(df["Dates"], dayfirst=True)

df = df.rename(columns={
    "PX_SETTLE": "Copper",
    "PX_SETTLE.1": "Aluminum",
    "PX_SETTLE.2": "Zinc",
    "PX_SETTLE.3": "Lead",
    "PX_SETTLE.4": "Tin",
    "PX_SETTLE.5": "CL_Future"
})

In [3]:
filtered_data = df.filter(items=["Dates", "Copper", "Zinc"])

filtered_data["Dates"] = pd.to_datetime(filtered_data["Dates"], dayfirst=True)

filtered_data = filtered_data[(filtered_data["Dates"] >= "01/01/2020") & (filtered_data["Dates"] <= "31/12/2021")]

filtered_data

,Dates,Copper,Zinc
2608,2020-01-01,6174.0,2272.0
2609,2020-01-02,6188.0,2310.0
2610,2020-01-03,6129.5,2306.0
2611,2020-01-06,6138.5,2324.5
2612,2020-01-07,6149.0,2346.0
...,...,...,...
3126,2021-12-27,9568.0,3519.0
3127,2021-12-28,9568.0,3519.0
3128,2021-12-29,9680.5,3513.0
3129,2021-12-30,9691.5,3532.5


Calculating MACD (slow/medium/fast) and RSI for each metal historically:

MACD:

Slow:

In [4]:
copper_slow_short_term_EMA = filtered_data["Copper"].ewm(span=24).mean()

copper_slow_longer_term_EMA = filtered_data["Copper"].ewm(span=52).mean()

copper_slow_MACD = copper_slow_short_term_EMA - copper_slow_longer_term_EMA

In [5]:
zinc_slow_short_term_EMA = filtered_data["Zinc"].ewm(span=24).mean()

zinc_slow_longer_term_EMA = filtered_data["Zinc"].ewm(span=52).mean()

zinc_slow_MACD = zinc_slow_short_term_EMA - zinc_slow_longer_term_EMA

Medium:

In [6]:
copper_medium_short_term_EMA = filtered_data["Copper"].ewm(span=12).mean()

copper_medium_longer_term_EMA = filtered_data["Copper"].ewm(span=26).mean()

copper_medium_MACD = copper_medium_short_term_EMA - copper_medium_longer_term_EMA

In [7]:
zinc_medium_short_term_EMA = filtered_data["Zinc"].ewm(span=12).mean()

zinc_medium_longer_term_EMA = filtered_data["Zinc"].ewm(span=26).mean()

zinc_medium_MACD = zinc_medium_short_term_EMA - zinc_medium_longer_term_EMA

Fast:

In [8]:
copper_fast_short_term_EMA = filtered_data["Copper"].ewm(span=6).mean()

copper_fast_longer_term_EMA = filtered_data["Copper"].ewm(span=13).mean()

copper_fast_MACD = copper_fast_short_term_EMA - copper_fast_longer_term_EMA

In [9]:
zinc_fast_short_term_EMA = filtered_data["Zinc"].ewm(span=6).mean()

zinc_fast_longer_term_EMA = filtered_data["Zinc"].ewm(span=13).mean()

zinc_fast_MACD = zinc_fast_short_term_EMA - zinc_fast_longer_term_EMA

RSI:

Copper:

In [10]:
copper_delta = filtered_data["Copper"].diff()

copper_gain = copper_delta.clip(lower=0)
copper_loss = -copper_delta.clip(upper=0)

copper_avg_gain = copper_gain.ewm(alpha=1/14, adjust=False).mean()
copper_avg_loss = copper_loss.ewm(alpha=1/14, adjust=False).mean()

copper_RS = copper_avg_gain / copper_avg_loss
copper_RSI = 100 - (100 / (1 + copper_RS))

Zinc:

In [11]:
zinc_delta = filtered_data["Zinc"].diff()

zinc_gain = zinc_delta.clip(lower=0)
zinc_loss = -zinc_delta.clip(upper=0)

zinc_avg_gain = zinc_gain.ewm(alpha=1/14, adjust=False).mean()
zinc_avg_loss = zinc_loss.ewm(alpha=1/14, adjust=False).mean()

zinc_RS = zinc_avg_gain / zinc_avg_loss
zinc_RSI = 100 - (100 / (1 + zinc_RS))

Using SQL inserts to populate the SQL table created in Question 2 with this generated data:

In [12]:
connection = sqlite3.connect("metals.db")
cursor = connection.cursor()

In [13]:
cursor.execute("""
DELETE FROM MetalPrices;
""")

In [13]:
filtered_data["Copper_Slow_MACD"] = copper_slow_MACD
filtered_data["Copper_Medium_MACD"] = copper_medium_MACD
filtered_data["Copper_Fast_MACD"] = copper_fast_MACD

filtered_data["Zinc_Slow_MACD"] = zinc_slow_MACD
filtered_data["Zinc_Medium_MACD"] = zinc_medium_MACD
filtered_data["Zinc_Fast_MACD"] = zinc_fast_MACD

filtered_data["Copper_RSI"] = copper_RSI
filtered_data["Zinc_RSI"] = zinc_RSI

Demonstrating the use of a decorator to log the execution of the SQL inserts:

Decorator:

In [33]:
def log_execution(func):
    async def async_wrapper(dataframe):
        
        start = time.time()
        print("Starting DB query:")

        result = await func(dataframe)

        end = time.time()
        print(f"\n\nDB query finished in {end - start:.2f} seconds\n\n")

        return result
    return async_wrapper

Function to use the above Decorator:

Modifying Question 3 to write data to the database asynchronously:

In [34]:
@log_execution
async def async_write_data(filtered_data):

    filtered_data["Dates"] = filtered_data["Dates"].astype(str)

    rows = list(filtered_data[[
        "Dates",
        "Copper",
        "Zinc",
        "Copper_Slow_MACD",
        "Copper_Medium_MACD",
        "Copper_Fast_MACD",
        "Zinc_Slow_MACD",
        "Zinc_Medium_MACD",
        "Zinc_Fast_MACD",
        "Copper_RSI",
        "Zinc_RSI"
    ]].itertuples(index=False, name=None))

    async with aiosqlite.connect("metals.db", timeout=10) as db:

        await db.execute("PRAGMA journal_mode=WAL;")

        # insert new data
        await db.executemany("""
            INSERT INTO MetalPrices (
                Dates, Copper, Zinc,
                Copper_Slow_MACD, Copper_Medium_MACD, Copper_Fast_MACD,
                Zinc_Slow_MACD, Zinc_Medium_MACD, Zinc_Fast_MACD,
                Copper_RSI, Zinc_RSI
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, rows)

        await db.commit()

In [17]:
task = asyncio.create_task(async_write_data(filtered_data))


copper_mean = filtered_data["Copper"].mean()
zinc_mean = filtered_data["Zinc"].mean()
print("Generating analysis...")
print(f"Average Copper Price: {copper_mean:.2f}")
print(f"Average Zinc Price: {zinc_mean:.2f}")

await task

Generating analysis...
Average Copper Price: 7740.68
Average Zinc Price: 2643.34
Starting DB insert...


OperationalError: database is locked

Reading from the database 5 times concurrantly using async:

In [35]:
@log_execution
async def read_data(run_id):

    async with aiosqlite.connect("metals.db") as db:

        print(f"[READ {run_id}] Starting read query:")

        async with db.execute("SELECT * FROM MetalPrices") as cursor:

            rows = await cursor.fetchall()

            print(f"\nREAD {run_id}:\n")
            print(f"First 5 rows fetched from this Read:")
            print(rows[:5])

            print(f"This Read has now Fetched {len(rows)} rows")

            return rows

In [36]:
results = await asyncio.gather(
    read_data(1),
    read_data(2),
    read_data(3),
    read_data(4),
    read_data(5)
)

print("5 concurrent reads completed")

Starting DB query:
Starting DB query:
Starting DB query:
Starting DB query:
Starting DB query:
[READ 1] Starting read query:
[READ 2] Starting read query:
[READ 3] Starting read query:
[READ 4] Starting read query:
[READ 5] Starting read query:

READ 2:

First 5 rows fetched from this Read:
[('2020-01-01 00:00:00', 6174.0, 2272.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, None, None), ('2020-01-02 00:00:00', 6188.0, 2310.0, 0.15705128205081564, 0.3141025641016313, 0.6282051282050816, 0.4262820512817598, 0.8525641025635196, 1.7051282051279486, 100.0, 100.0), ('2020-01-03 00:00:00', 6129.5, 2306.0, -0.6869506679695405, -1.413852487659824, -2.9698042331856414, 0.49531141852048677, 0.9622961287659564, 1.8011991620314802, 75.67567567567568, 99.19678714859438), ('2020-01-06 00:00:00', 6138.5, 2324.5, -0.925829457592954, -1.8401688124531574, -3.5643400187518637, 0.8461054994982078, 1.6565564241659558, 3.139488300517769, 76.61798616448885, 99.22768453883856), ('2020-01-07 00:00:00', 6149.0, 2346.0, -0.854

In [37]:
resultant_df = pd.read_sql("SELECT * FROM MetalPrices", connection)

resultant_df

,Dates,Copper,Zinc,Copper_Slow_MACD,Copper_Medium_MACD,Copper_Fast_MACD,Zinc_Slow_MACD,Zinc_Medium_MACD,Zinc_Fast_MACD,Copper_RSI,Zinc_RSI
0,2020-01-01 00:00:00,6174.0,2272.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
1,2020-01-02 00:00:00,6188.0,2310.0,0.157051,0.314103,0.628205,0.426282,0.852564,1.705128,100.000000,100.000000
2,2020-01-03 00:00:00,6129.5,2306.0,-0.686951,-1.413852,-2.969804,0.495311,0.962296,1.801199,75.675676,99.196787
3,2020-01-06 00:00:00,6138.5,2324.5,-0.925829,-1.840169,-3.564340,0.846105,1.656556,3.139488,76.617986,99.227685
4,2020-01-07 00:00:00,6149.0,2346.0,-0.854880,-1.615263,-2.767289,1.447627,2.854467,5.441286,77.703228,99.263159
...,...,...,...,...,...,...,...,...,...,...,...
518,2021-12-27 00:00:00,9568.0,3519.0,-6.169480,-2.500190,24.249359,61.833178,66.849430,64.568122,51.692406,66.260897
519,2021-12-28 00:00:00,9568.0,3519.0,-4.327724,0.815513,23.108353,66.019887,69.361656,60.552208,51.692406,66.260897
520,2021-12-29 00:00:00,9680.5,3513.0,2.070534,12.378352,37.538009,69.273420,70.060845,54.764823,56.618811,65.418907
521,2021-12-30 00:00:00,9691.5,3532.5,8.193457,22.173985,46.411756,72.768741,71.365785,51.771942,57.079706,66.891416
